# Đồ thị Cross-sell: Mối quan hệ mua chéo giữa các nhóm danh mục

## Bối cảnh kinh doanh

Trong hệ thống gợi ý sản phẩm, nếu chỉ giới thiệu sản phẩm cùng loại (same-category) thì chỉ mang lại sự **tiện lợi** cho khách hàng, chứ chưa tối ưu **doanh thu**.

**Cross-sell** — gợi ý sản phẩm từ danh mục KHÁC — mới thực sự tạo ra tăng trưởng doanh thu. Notebook này trực quan hóa mối quan hệ mua chéo giữa các nhóm danh mục để hỗ trợ chiến lược cross-sell.

## Ngữ cảnh dự án (đã xác minh)

| Thông tin | Chi tiết |
|---|---|
| **Repo** | PJ-SELLING-WEBSITE (CS116) |
| **Notebook đang chỉnh sửa** | `Additional Doc/Related/graph.ipynb` |
| **Dữ liệu nguồn** | `raw_data/items.parquet` (29,823 items), `raw_data/transactions-202411-to-202412.parquet` (~6M rows), `raw_data/transactions-2025-12.parquet` (~3.7M rows) |
| **Thư viện** | `polars` (tiền xử lý), `pyvis` (trực quan hóa đồ thị) |

## Giả định tiền xử lý

1. **`sale_status == 1` only**: Chỉ giữ các sản phẩm đang bán (`sale_status == 1`). Bỏ toàn bộ `sale_status == 0`.
2. **Pseudo-session**: Vì dữ liệu không có `order_id` / `cart_id`, repo sử dụng cách tiếp cận pseudo-session:
   - `session = (customer_id, updated_date.dt.date())`
   - Giữ session có 2–50 item duy nhất
3. **Ghép dọc**: 2 file transaction được merge theo chiều dọc (vertical concat).
4. **Cross-sell**: Chỉ tạo cạnh giữa các danh mục KHÁC nhau (không có self-loop).

## Quy tắc đặc biệt: Sữa và Tã

### Trong đồ thị L1+L2
- `Sữa` và `Tã` **không** được tách ra thành `L1 | L2` vì:
  - L2 của Sữa là tên **thương hiệu** (Abbott, Vinamilk, ...), không phải phân loại sản phẩm
  - L2 của Tã cũng là tên **thương hiệu** (Huggies, Bobby, ...)
  - Tách ra sẽ tạo quá nhiều node vô nghĩa, làm đồ thị khó đọc
- Tất cả danh mục khác sử dụng label `L1 | L2` bình thường

### Loại bỏ cạnh Sữa / Tã (cả 2 đồ thị)
- `Sữa` và `Tã` là 2 danh mục chiếm tỉ trọng rất lớn → tạo ra rất nhiều cạnh mạnh, chiếm ưu thế và "nuốt" hết các mối quan hệ khác
- Để đồ thị thể hiện rõ các mối quan hệ cross-sell **giữa các danh mục còn lại**, ta **loại bỏ tất cả cạnh** liên quan đến Sữa và Tã
- Tuy nhiên, 2 node này **vẫn được hiển thị** trên đồ thị dưới dạng **node cô lập** (isolated) để người xem biết chúng tồn tại

## Về ngưỡng lọc `S > 100`

Công thức tính điểm: `S = ln(count_ab) * (P(B|A) + P(A|B))`

Vì `P(B|A)` và `P(A|B)` đều nằm trong khoảng `[0, 1]`, nên `P(B|A) + P(A|B) ≤ 2`. Do đó giá trị tối đa của `S` là `ln(count_ab) * 2`. Với `count_ab` hàng trăm nghìn, `ln(count_ab)` chỉ khoảng 10–12, nên `S` tối đa chỉ khoảng **20–24**.

→ **`MIN_SCORE = 100` là không khả thi** và sẽ làm đồ thị trống hoàn toàn.

**Chiến lược fallback đã triển khai:**
- Giữ nguyên `MIN_COUNT_AB = 100` (lọc cặp có ít nhất 100 co-occurrence)
- Giữ top 2 neighbor theo score cho mỗi node
- `MIN_SCORE = 100` được khai báo nhưng có cờ fallback tự động khi không khả thi

In [37]:
# ============================================================
# Cài đặt thư viện cần thiết (nếu chưa có)
# ============================================================
%pip install pyvis polars --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
# ============================================================
# Import thư viện
# ============================================================
import polars as pl
import math
import os
from pathlib import Path
from itertools import combinations
from collections import Counter

from pyvis.network import Network
from IPython.display import display, HTML

In [39]:
# ============================================================
# Hằng số cấu hình
# ============================================================

# Đường dẫn dữ liệu gốc (tương đối từ thư mục notebook)
ROOT = Path("../..")
RAW_DIR = ROOT / "raw_data"

# Tên file dữ liệu giao dịch
TRANSACTION_FILES = [
    RAW_DIR / "transactions-202411-to-202412.parquet",
    RAW_DIR / "transactions-2025-12.parquet",
]
ITEMS_FILE = RAW_DIR / "items.parquet"

# Ngưỡng lọc session (theo convention của repo)
MIN_SESSION_ITEMS = 2    # Tối thiểu 2 item duy nhất trong 1 session
MAX_SESSION_ITEMS = 50   # Tối đa 50 item (loại bỏ bulk buyer / nhiễu)

# Ngưỡng lọc cạnh đồ thị
MIN_COUNT_AB = 100       # Số session chứa cả A và B ít nhất 100
MIN_SCORE = 100          # Ngưỡng điểm tối thiểu (xem giải thích ở trên)
TOP_K_NEIGHBORS = 2      # Giữ top K neighbor theo score cho mỗi node

# Các danh mục giữ nguyên ở cấp L1 trong đồ thị L1+L2
KEEP_L1_ONLY_CATEGORIES = {"Sữa", "Tã"}

# Các danh mục hiển thị dạng node cô lập (không có cạnh)
# Sữa và Tã chiếm tỉ trọng quá lớn, loại cạnh để đồ thị rõ ràng hơn
ISOLATED_CATEGORIES = {"Sữa", "Tã"}

print("✅ Đã nạp hằng số cấu hình")
print(f"   Thư mục gốc: {ROOT.resolve()}")
print(f"   File items: {ITEMS_FILE}")
print(f"   File transactions: {[str(f) for f in TRANSACTION_FILES]}")
print(f"   Node cô lập (giữ node, bỏ cạnh): {ISOLATED_CATEGORIES}")

✅ Đã nạp hằng số cấu hình
   Thư mục gốc: C:\Users\asus\Documents\HocTap\HK4\CS116 - Python Programming For ML\Pj-selling website
   File items: ..\..\raw_data\items.parquet
   File transactions: ['..\\..\\raw_data\\transactions-202411-to-202412.parquet', '..\\..\\raw_data\\transactions-2025-12.parquet']
   Node cô lập (giữ node, bỏ cạnh): {'Sữa', 'Tã'}


---
## 1. Nạp và tiền xử lý dữ liệu

In [40]:
# ============================================================
# 1.1 Nạp dữ liệu sản phẩm — chỉ lấy cột cần thiết
# ============================================================

# Chỉ đọc các cột cần thiết để tiết kiệm bộ nhớ
items_needed_cols = ["item_id", "category_l1", "category_l2", "sale_status"]

items = (
    pl.scan_parquet(ITEMS_FILE)
    .select(items_needed_cols)
    .filter(pl.col("sale_status") == 1)        # Chỉ giữ sản phẩm đang bán
    .drop("sale_status")                        # Không cần nữa sau khi lọc
    .collect()
)

print(f"📦 Số sản phẩm đang bán (sale_status=1): {items.height:,}")
print(f"   Số category L1 duy nhất: {items['category_l1'].n_unique()}")
print(f"   Số category L1+L2 duy nhất: {items.select(pl.concat_str(['category_l1', 'category_l2'], separator=' | ')).to_series().n_unique()}")

📦 Số sản phẩm đang bán (sale_status=1): 6,850
   Số category L1 duy nhất: 15
   Số category L1+L2 duy nhất: 111


In [41]:
# ============================================================
# 1.2 Nạp và ghép dọc 2 file giao dịch
# ============================================================

# Chỉ cần customer_id, item_id, updated_date cho bước tạo session
txn_needed_cols = ["customer_id", "item_id", "updated_date"]

txn_frames = []
for f in TRANSACTION_FILES:
    df = (
        pl.scan_parquet(f)
        .select(txn_needed_cols)
        .collect()
    )
    print(f"  📄 {f.name}: {df.height:,} dòng")
    txn_frames.append(df)

# Ghép dọc (vertical concat)
transactions = pl.concat(txn_frames, how="vertical_relaxed")
print(f"\n📊 Tổng giao dịch sau ghép dọc: {transactions.height:,} dòng")

# Giải phóng bộ nhớ
del txn_frames

  📄 transactions-202411-to-202412.parquet: 6,053,447 dòng
  📄 transactions-2025-12.parquet: 3,782,447 dòng

📊 Tổng giao dịch sau ghép dọc: 9,835,894 dòng


In [42]:
# ============================================================
# 1.3 Lọc giao dịch — chỉ giữ item đang bán
# ============================================================

# Tập item_id hợp lệ (sale_status == 1)
active_item_ids = items["item_id"].unique()

transactions_before = transactions.height
transactions = transactions.filter(
    pl.col("item_id").is_in(active_item_ids.implode())
)

print(f"🔍 Giao dịch trước lọc: {transactions_before:,}")
print(f"   Giao dịch sau lọc (chỉ item sale_status=1): {transactions.height:,}")
print(f"   Đã loại: {transactions_before - transactions.height:,} dòng")

🔍 Giao dịch trước lọc: 9,835,894
   Giao dịch sau lọc (chỉ item sale_status=1): 7,933,245
   Đã loại: 1,902,649 dòng


In [43]:
# ============================================================
# 1.4 Join giao dịch với danh mục sản phẩm
# ============================================================

# Join để lấy category_l1, category_l2 cho mỗi giao dịch
txn_with_cat = transactions.join(
    items.select(["item_id", "category_l1", "category_l2"]),
    on="item_id",
    how="inner"
)

print(f"🔗 Giao dịch sau join với danh mục: {txn_with_cat.height:,} dòng")
print(f"   Cột: {txn_with_cat.columns}")

# Giải phóng bộ nhớ
del transactions

🔗 Giao dịch sau join với danh mục: 7,933,245 dòng
   Cột: ['customer_id', 'item_id', 'updated_date', 'category_l1', 'category_l2']


---
## 2. Xây dựng pseudo-session

In [44]:
# ============================================================
# 2.1 Tạo pseudo-session theo (customer_id, session_date)
# ============================================================

# Thêm cột ngày từ updated_date
txn_with_cat = txn_with_cat.with_columns(
    pl.col("updated_date").dt.date().alias("session_date")
)

# Đếm số item duy nhất trong mỗi session
session_item_counts = (
    txn_with_cat
    .group_by(["customer_id", "session_date"])
    .agg(pl.col("item_id").n_unique().alias("n_unique_items"))
)

# Lọc session có 2–50 item duy nhất (theo convention repo)
valid_sessions = session_item_counts.filter(
    (pl.col("n_unique_items") >= MIN_SESSION_ITEMS) &
    (pl.col("n_unique_items") <= MAX_SESSION_ITEMS)
)

print(f"📋 Tổng số session: {session_item_counts.height:,}")
print(f"   Session hợp lệ (2–50 item): {valid_sessions.height:,}")
print(f"   Session bị loại: {session_item_counts.height - valid_sessions.height:,}")

📋 Tổng số session: 3,522,144
   Session hợp lệ (2–50 item): 1,730,045
   Session bị loại: 1,792,099


In [45]:
# ============================================================
# 2.2 Lọc giao dịch chỉ thuộc session hợp lệ
# ============================================================

txn_valid = txn_with_cat.join(
    valid_sessions.select(["customer_id", "session_date"]),
    on=["customer_id", "session_date"],
    how="semi"
)

print(f"✅ Giao dịch trong session hợp lệ: {txn_valid.height:,} dòng")

# Giải phóng bộ nhớ
del txn_with_cat, session_item_counts

✅ Giao dịch trong session hợp lệ: 6,109,675 dòng


---
## 3. Tạo cặp danh mục co-buy (cross-sell)

In [46]:
# ============================================================
# 3.1 Hàm tạo label danh mục cho đồ thị L1+L2
# ============================================================

def build_effective_category_label(category_l1: str, category_l2: str) -> str:
    """
    Tạo nhãn danh mục hiệu quả cho đồ thị L1+L2.

    Quy tắc đặc biệt:
    - Nếu category_l1 là "Sữa" hoặc "Tã" → giữ nguyên L1 làm nhãn
      (vì L2 của chúng là tên thương hiệu, không phải phân loại sản phẩm)
    - Các danh mục khác → sử dụng "L1 | L2" làm nhãn
    """
    if category_l1 in KEEP_L1_ONLY_CATEGORIES:
        return category_l1
    return f"{category_l1} | {category_l2}"

print("✅ Đã định nghĩa hàm build_effective_category_label()")
print(f"   Danh mục giữ nguyên L1: {KEEP_L1_ONLY_CATEGORIES}")
print(f"   Ví dụ: build_effective_category_label('Sữa', 'Abbott') → '{build_effective_category_label('Sữa', 'Abbott')}'")
print(f"   Ví dụ: build_effective_category_label('Babycare', 'Bé ngủ') → '{build_effective_category_label('Babycare', 'Bé ngủ')}'")

✅ Đã định nghĩa hàm build_effective_category_label()
   Danh mục giữ nguyên L1: {'Sữa', 'Tã'}
   Ví dụ: build_effective_category_label('Sữa', 'Abbott') → 'Sữa'
   Ví dụ: build_effective_category_label('Babycare', 'Bé ngủ') → 'Babycare | Bé ngủ'


In [47]:
# ============================================================
# 3.2 Hàm chung: tạo cặp danh mục co-buy từ session
# ============================================================

def build_category_pairs(
    txn_df: pl.DataFrame,
    category_col: str,
    graph_label: str
) -> pl.DataFrame:
    """
    Tạo cặp danh mục co-buy từ dữ liệu giao dịch.

    Quy trình:
    1. Gom các danh mục duy nhất trong mỗi session
    2. Lọc session có ≥ 2 danh mục khác nhau (cross-sell)
    3. Tạo tất cả cặp danh mục (không có self-loop)
    4. Đếm tần suất xuất hiện của mỗi cặp

    Args:
        txn_df: DataFrame giao dịch có cột [customer_id, session_date, category_col]
        category_col: Tên cột danh mục để nhóm (ví dụ: 'category_l1' hoặc 'cat_label')
        graph_label: Nhãn mô tả đồ thị (để in log)

    Returns:
        DataFrame với cột [cat_a, cat_b, count_ab, count_a, count_b]
    """
    print(f"\n🔧 Đang tạo cặp cho đồ thị: {graph_label}")

    # Bước 1: Gom danh mục duy nhất trong mỗi session
    sessions_cats = (
        txn_df
        .group_by(["customer_id", "session_date"])
        .agg(pl.col(category_col).unique().alias("categories"))
        .filter(pl.col("categories").list.len() >= 2)  # Cần ≥ 2 danh mục khác nhau
    )

    n_sessions = sessions_cats.height
    print(f"   Session có ≥ 2 danh mục khác nhau: {n_sessions:,}")

    # Bước 2: Tạo cặp danh mục và đếm
    pair_counts: Counter = Counter()
    cat_session_counts: Counter = Counter()  # Đếm số session chứa mỗi danh mục

    for row in sessions_cats.iter_rows(named=True):
        cats = sorted(set(row["categories"]))
        # Đếm số session của mỗi danh mục
        for cat in cats:
            cat_session_counts[cat] += 1
        # Tạo cặp (cross-sell: chỉ giữa danh mục khác nhau)
        for a, b in combinations(cats, 2):
            pair_counts[(a, b)] += 1

    print(f"   Số cặp danh mục duy nhất: {len(pair_counts):,}")

    # Bước 3: Chuyển sang DataFrame
    if not pair_counts:
        return pl.DataFrame({
            "cat_a": pl.Series([], dtype=pl.Utf8),
            "cat_b": pl.Series([], dtype=pl.Utf8),
            "count_ab": pl.Series([], dtype=pl.Int64),
            "count_a": pl.Series([], dtype=pl.Int64),
            "count_b": pl.Series([], dtype=pl.Int64),
        })

    cats_a, cats_b, counts = [], [], []
    counts_a_list, counts_b_list = [], []
    for (a, b), count_ab in pair_counts.items():
        cats_a.append(a)
        cats_b.append(b)
        counts.append(count_ab)
        counts_a_list.append(cat_session_counts[a])
        counts_b_list.append(cat_session_counts[b])

    pairs_df = pl.DataFrame({
        "cat_a": cats_a,
        "cat_b": cats_b,
        "count_ab": counts,
        "count_a": counts_a_list,
        "count_b": counts_b_list,
    })

    return pairs_df

print("✅ Đã định nghĩa hàm build_category_pairs()")

✅ Đã định nghĩa hàm build_category_pairs()


In [48]:
# ============================================================
# 3.3 Tạo cặp cho đồ thị Category L1
# ============================================================

pairs_l1 = build_category_pairs(
    txn_valid,
    category_col="category_l1",
    graph_label="Category L1"
)

print(f"\n📊 Kết quả cặp L1: {pairs_l1.height} cặp")


🔧 Đang tạo cặp cho đồ thị: Category L1
   Session có ≥ 2 danh mục khác nhau: 1,327,551
   Số cặp danh mục duy nhất: 105

📊 Kết quả cặp L1: 105 cặp


In [49]:
# ============================================================
# 3.4 Tạo cặp cho đồ thị Category L1+L2
# ============================================================

# Thêm cột nhãn danh mục hiệu quả (áp dụng quy tắc Sữa/Tã)
txn_with_label = txn_valid.with_columns(
    pl.struct(["category_l1", "category_l2"])
    .map_elements(
        lambda row: build_effective_category_label(row["category_l1"], row["category_l2"]),
        return_dtype=pl.Utf8
    )
    .alias("cat_label")
)

pairs_l1l2 = build_category_pairs(
    txn_with_label,
    category_col="cat_label",
    graph_label="Category L1+L2"
)

print(f"\n📊 Kết quả cặp L1+L2: {pairs_l1l2.height} cặp")

# Giải phóng bộ nhớ
del txn_with_label, txn_valid


🔧 Đang tạo cặp cho đồ thị: Category L1+L2
   Session có ≥ 2 danh mục khác nhau: 1,498,078
   Số cặp danh mục duy nhất: 2,755

📊 Kết quả cặp L1+L2: 2755 cặp


---
## 4. Tính điểm cạnh (scoring) và lọc

In [50]:
# ============================================================
# 4.1 Hàm tính điểm và lọc cạnh
# ============================================================

def compute_edge_scores(pairs_df: pl.DataFrame, graph_label: str) -> pl.DataFrame:
    """
    Tính điểm cạnh theo công thức:
        S = ln(count_ab) * (P(B|A) + P(A|B))

    Trong đó:
        - count_ab: số session chứa cả A và B
        - count_a: số session chứa A
        - count_b: số session chứa B
        - P(B|A) = count_ab / count_a
        - P(A|B) = count_ab / count_b

    Args:
        pairs_df: DataFrame với cột [cat_a, cat_b, count_ab, count_a, count_b]
        graph_label: Nhãn mô tả đồ thị (để in log)

    Returns:
        DataFrame bổ sung thêm cột [p_b_given_a, p_a_given_b, score]
    """
    print(f"\n📐 Tính điểm cho: {graph_label}")

    scored = pairs_df.with_columns([
        # P(B|A) = count_ab / count_a
        (pl.col("count_ab").cast(pl.Float64) / pl.col("count_a").cast(pl.Float64))
            .alias("p_b_given_a"),
        # P(A|B) = count_ab / count_b
        (pl.col("count_ab").cast(pl.Float64) / pl.col("count_b").cast(pl.Float64))
            .alias("p_a_given_b"),
    ]).with_columns(
        # S = ln(count_ab) * (P(B|A) + P(A|B))
        (pl.col("count_ab").cast(pl.Float64).log(base=math.e)
         * (pl.col("p_b_given_a") + pl.col("p_a_given_b")))
        .alias("score")
    ).sort("score", descending=True)

    print(f"   Phạm vi score: [{scored['score'].min():.4f}, {scored['score'].max():.4f}]")
    print(f"   Trung vị score: {scored['score'].median():.4f}")

    return scored


def filter_and_select_top_neighbors(
    scored_df: pl.DataFrame,
    min_count_ab: int,
    min_score: float,
    top_k: int,
    graph_label: str
) -> pl.DataFrame:
    """
    Lọc cạnh và chọn top-K neighbor cho mỗi node.

    Quy trình:
    1. Lọc count_ab > min_count_ab
    2. Kiểm tra MIN_SCORE — nếu không khả thi thì bỏ qua bước lọc score
    3. Với mỗi node, giữ top-K neighbor theo score
    4. Lấy union tất cả cạnh được chọn

    Args:
        scored_df: DataFrame đã tính score
        min_count_ab: Ngưỡng tối thiểu cho count_ab
        min_score: Ngưỡng tối thiểu cho score (có thể bị bỏ qua nếu không khả thi)
        top_k: Số neighbor tối đa cho mỗi node
        graph_label: Nhãn mô tả đồ thị

    Returns:
        DataFrame các cạnh được chọn
    """
    print(f"\n🔎 Lọc cạnh cho: {graph_label}")

    # Bước 1: Lọc theo count_ab
    filtered = scored_df.filter(pl.col("count_ab") > min_count_ab)
    print(f"   Sau lọc count_ab > {min_count_ab}: {filtered.height} cạnh")

    # Bước 2: Kiểm tra ngưỡng MIN_SCORE
    if filtered.height > 0:
        max_score = filtered["score"].max()
        if max_score < min_score:
            print(f"   ⚠️  Score tối đa ({max_score:.4f}) < MIN_SCORE ({min_score})")
            print(f"   ⚠️  Ngưỡng MIN_SCORE = {min_score} KHÔNG KHẢ THI với thang điểm thực tế!")
            print(f"   ⚠️  Công thức S = ln(count_ab) * (P(B|A) + P(A|B)) có giá trị tối đa lý thuyết ≈ ln(N)*2")
            print(f"   ⚠️  → BỎ QUA ngưỡng MIN_SCORE, chỉ dùng count_ab > {min_count_ab} + top-{top_k} neighbor")
        else:
            filtered = filtered.filter(pl.col("score") > min_score)
            print(f"   Sau lọc score > {min_score}: {filtered.height} cạnh")

    if filtered.height == 0:
        print(f"   ⚠️  Không còn cạnh nào sau lọc!")
        return filtered

    # Bước 3: Với mỗi node, chọn top-K neighbor theo score
    # Mỗi cạnh (A, B) thuộc về cả node A và node B
    # Ta cần xem xét từ cả 2 phía
    col_order = filtered.columns  # Lưu thứ tự cột gốc để đảm bảo nhất quán khi concat

    # Phía A: top-K neighbor từ góc nhìn của cat_a
    top_from_a = (
        filtered
        .sort("score", descending=True)
        .group_by("cat_a")
        .head(top_k)
        .select(col_order)  # Đảm bảo thứ tự cột nhất quán
    )

    # Phía B: top-K neighbor từ góc nhìn của cat_b
    top_from_b = (
        filtered
        .sort("score", descending=True)
        .group_by("cat_b")
        .head(top_k)
        .select(col_order)  # Đảm bảo thứ tự cột nhất quán
    )

    # Union: lấy tập hợp tất cả cạnh được chọn bởi ít nhất 1 node
    # Loại trùng lặp theo cặp (cat_a, cat_b)
    selected = pl.concat([top_from_a, top_from_b]).unique(
        subset=["cat_a", "cat_b"]
    ).sort("score", descending=True)

    print(f"   Sau chọn top-{top_k} neighbor: {selected.height} cạnh")

    return selected


def remove_isolated_category_edges(
    edges_df: pl.DataFrame,
    isolated_cats: set[str],
    graph_label: str
) -> pl.DataFrame:
    """
    Loại bỏ tất cả cạnh liên quan đến các danh mục cô lập.
    Các danh mục này vẫn được hiển thị dưới dạng node cô lập trong đồ thị.

    Args:
        edges_df: DataFrame cạnh đã lọc
        isolated_cats: Tập danh mục cần cô lập
        graph_label: Nhãn mô tả đồ thị (để in log)

    Returns:
        DataFrame cạnh sau khi loại bỏ
    """
    if not isolated_cats or edges_df.height == 0:
        return edges_df

    before = edges_df.height
    cleaned = edges_df.filter(
        ~pl.col("cat_a").is_in(list(isolated_cats)) &
        ~pl.col("cat_b").is_in(list(isolated_cats))
    )
    removed = before - cleaned.height
    print(f"   🚫 Đã loại {removed} cạnh liên quan đến {isolated_cats} — còn {cleaned.height} cạnh")

    return cleaned


print("✅ Đã định nghĩa các hàm scoring và lọc")

✅ Đã định nghĩa các hàm scoring và lọc


In [51]:
# ============================================================
# 4.2 Áp dụng scoring + lọc cho đồ thị L1
# ============================================================

scored_l1 = compute_edge_scores(pairs_l1, "Category L1")
edges_l1 = filter_and_select_top_neighbors(
    scored_l1, MIN_COUNT_AB, MIN_SCORE, TOP_K_NEIGHBORS, "Category L1"
)

# Loại bỏ cạnh Sữa / Tã — giữ node cô lập
edges_l1 = remove_isolated_category_edges(edges_l1, ISOLATED_CATEGORIES, "Category L1")

print(f"\n📊 Đồ thị L1 — Kết quả cuối cùng:")
print(f"   Số cạnh: {edges_l1.height}")
nodes_l1 = set(edges_l1["cat_a"].to_list()) | set(edges_l1["cat_b"].to_list())
print(f"   Số node (có cạnh): {len(nodes_l1)}")
print(f"   Số node cô lập (Sữa, Tã): {len(ISOLATED_CATEGORIES)}")
print(f"   Tổng node hiển thị: {len(nodes_l1) + len(ISOLATED_CATEGORIES)}")


📐 Tính điểm cho: Category L1
   Phạm vi score: [0.1978, 9.1694]
   Trung vị score: 2.5995

🔎 Lọc cạnh cho: Category L1
   Sau lọc count_ab > 100: 105 cạnh
   ⚠️  Score tối đa (9.1694) < MIN_SCORE (100)
   ⚠️  Ngưỡng MIN_SCORE = 100 KHÔNG KHẢ THI với thang điểm thực tế!
   ⚠️  Công thức S = ln(count_ab) * (P(B|A) + P(A|B)) có giá trị tối đa lý thuyết ≈ ln(N)*2
   ⚠️  → BỎ QUA ngưỡng MIN_SCORE, chỉ dùng count_ab > 100 + top-2 neighbor
   Sau chọn top-2 neighbor: 46 cạnh
   🚫 Đã loại 11 cạnh liên quan đến {'Sữa', 'Tã'} — còn 35 cạnh

📊 Đồ thị L1 — Kết quả cuối cùng:
   Số cạnh: 35
   Số node (có cạnh): 13
   Số node cô lập (Sữa, Tã): 2
   Tổng node hiển thị: 15


In [52]:
# ============================================================
# 4.3 Áp dụng scoring + lọc cho đồ thị L1+L2
# ============================================================

scored_l1l2 = compute_edge_scores(pairs_l1l2, "Category L1+L2")
edges_l1l2 = filter_and_select_top_neighbors(
    scored_l1l2, MIN_COUNT_AB, MIN_SCORE, TOP_K_NEIGHBORS, "Category L1+L2"
)

# Loại bỏ cạnh Sữa / Tã — giữ node cô lập
edges_l1l2 = remove_isolated_category_edges(edges_l1l2, ISOLATED_CATEGORIES, "Category L1+L2")

print(f"\n📊 Đồ thị L1+L2 — Kết quả cuối cùng:")
print(f"   Số cạnh: {edges_l1l2.height}")
nodes_l1l2 = set(edges_l1l2["cat_a"].to_list()) | set(edges_l1l2["cat_b"].to_list())
print(f"   Số node (có cạnh): {len(nodes_l1l2)}")
print(f"   Số node cô lập (Sữa, Tã): {len(ISOLATED_CATEGORIES)}")
print(f"   Tổng node hiển thị: {len(nodes_l1l2) + len(ISOLATED_CATEGORIES)}")


📐 Tính điểm cho: Category L1+L2
   Phạm vi score: [0.0000, 5.8046]
   Trung vị score: 0.3069

🔎 Lọc cạnh cho: Category L1+L2
   Sau lọc count_ab > 100: 1813 cạnh
   ⚠️  Score tối đa (5.8046) < MIN_SCORE (100)
   ⚠️  Ngưỡng MIN_SCORE = 100 KHÔNG KHẢ THI với thang điểm thực tế!
   ⚠️  Công thức S = ln(count_ab) * (P(B|A) + P(A|B)) có giá trị tối đa lý thuyết ≈ ln(N)*2
   ⚠️  → BỎ QUA ngưỡng MIN_SCORE, chỉ dùng count_ab > 100 + top-2 neighbor
   Sau chọn top-2 neighbor: 239 cạnh
   🚫 Đã loại 60 cạnh liên quan đến {'Sữa', 'Tã'} — còn 179 cạnh

📊 Đồ thị L1+L2 — Kết quả cuối cùng:
   Số cạnh: 179
   Số node (có cạnh): 69
   Số node cô lập (Sữa, Tã): 2
   Tổng node hiển thị: 71


---
## 5. Sanity Check — Kiểm tra dữ liệu

In [53]:
# ============================================================
# 5.1 Xem trước cạnh có score cao nhất — L1
# ============================================================

print("🏆 Top 10 cạnh score cao nhất — Category L1 (sau loại Sữa/Tã):")
print("=" * 80)
preview_l1 = edges_l1.head(10).select([
    "cat_a", "cat_b", "count_ab", "count_a", "count_b",
    "p_b_given_a", "p_a_given_b", "score"
])
for row in preview_l1.iter_rows(named=True):
    print(f"  {row['cat_a']:25s} ↔ {row['cat_b']:25s}  "
          f"count_ab={row['count_ab']:>6,}  "
          f"P(B|A)={row['p_b_given_a']:.4f}  "
          f"P(A|B)={row['p_a_given_b']:.4f}  "
          f"score={row['score']:.4f}")

🏆 Top 10 cạnh score cao nhất — Category L1 (sau loại Sữa/Tã):
  Sữa nước                  ↔ Thực phẩm cho bé           count_ab=151,333  P(B|A)=0.5612  P(A|B)=0.2076  score=9.1694
  Thực phẩm cho bé          ↔ Thực phẩm cho gia đình     count_ab=86,170  P(B|A)=0.1182  P(A|B)=0.6020  score=8.1848
  Hóa mỹ phẩm cho bé        ↔ Thực phẩm cho bé           count_ab=162,906  P(B|A)=0.4429  P(A|B)=0.2235  score=7.9978
  Babycare                  ↔ Hóa mỹ phẩm cho bé         count_ab=123,090  P(B|A)=0.3366  P(A|B)=0.3347  score=7.8677
  Babycare                  ↔ Thực phẩm cho bé           count_ab=153,717  P(B|A)=0.4203  P(A|B)=0.2109  score=7.5387
  Thực phẩm cho bé          ↔ Đồ chơi & Sách             count_ab=68,231  P(B|A)=0.0936  P(A|B)=0.5043  score=6.6552
  TPCN                      ↔ Thực phẩm cho bé           count_ab=81,542  P(B|A)=0.4739  P(A|B)=0.1119  score=6.6248
  Thực phẩm cho bé          ↔ Vệ sinh                    count_ab=100,288  P(B|A)=0.1376  P(A|B)=0.4175  score=6.39

In [54]:
# ============================================================
# 5.2 Xem trước cạnh có score cao nhất — L1+L2
# ============================================================

print("🏆 Top 10 cạnh score cao nhất — Category L1+L2 (sau loại Sữa/Tã):")
print("=" * 100)
preview_l1l2 = edges_l1l2.head(10).select([
    "cat_a", "cat_b", "count_ab", "count_a", "count_b",
    "p_b_given_a", "p_a_given_b", "score"
])
for row in preview_l1l2.iter_rows(named=True):
    print(f"  {row['cat_a']:40s} ↔ {row['cat_b']:40s}")
    print(f"      count_ab={row['count_ab']:>6,}  "
          f"P(B|A)={row['p_b_given_a']:.4f}  "
          f"P(A|B)={row['p_a_given_b']:.4f}  "
          f"score={row['score']:.4f}")

🏆 Top 10 cạnh score cao nhất — Category L1+L2 (sau loại Sữa/Tã):
  Thực phẩm cho bé | Snack ăn dặm          ↔ Thực phẩm cho bé | TP từ sữa (bảo quản lạnh)
      count_ab=73,036  P(B|A)=0.2340  P(A|B)=0.2844  score=5.8046
  Thực phẩm cho bé | Dầu ăn & Gia vị       ↔ Thực phẩm cho bé | Mì & Đồ khô ăn liền  
      count_ab=42,741  P(B|A)=0.2405  P(A|B)=0.2879  score=5.6344
  Thực phẩm cho bé | Dầu ăn & Gia vị       ↔ Thực phẩm cho bé | Snack ăn dặm         
      count_ab=54,749  P(B|A)=0.3081  P(A|B)=0.1754  score=5.2749
  Babycare | Đồ dùng vệ sinh               ↔ Hóa mỹ phẩm cho bé | Vệ sinh cho bé     
      count_ab=37,186  P(B|A)=0.2647  P(A|B)=0.2323  score=5.2301
  Textile | Khăn em bé                     ↔ Thời trang | Quần áo & Phụ kiện sơ sinh 
      count_ab=11,789  P(B|A)=0.2538  P(A|B)=0.2991  score=5.1840
  Thực phẩm cho bé | Mì & Đồ khô ăn liền   ↔ Thực phẩm cho bé | Snack ăn dặm         
      count_ab=47,321  P(B|A)=0.3187  P(A|B)=0.1516  score=5.0631
  Textile | Chăn   

---
## 6. Trực quan hóa bằng Pyvis

In [55]:
# ============================================================
# 6.1 Hàm xây dựng đồ thị Pyvis
# ============================================================

def build_pyvis_graph(
    edges_df: pl.DataFrame,
    title: str,
    isolated_nodes: set[str] | None = None,
    height: str = "750px",
    width: str = "100%"
) -> Network:
    """
    Xây dựng đồ thị tương tác bằng pyvis.

    Đặc điểm:
    - Node = nhóm danh mục
    - Cạnh dày hơn, đậm hơn = mối liên hệ mạnh hơn
    - Hover tooltip hiển thị các chỉ số chi tiết
    - Physics layout cho tương tác kéo thả
    - Node cô lập (isolated_nodes) hiển thị riêng không có cạnh

    Args:
        edges_df: DataFrame với cột [cat_a, cat_b, count_ab, count_a, count_b,
                  p_b_given_a, p_a_given_b, score]
        title: Tiêu đề đồ thị
        isolated_nodes: Tập node cô lập cần hiển thị (không có cạnh)
        height: Chiều cao (CSS)
        width: Chiều rộng (CSS)

    Returns:
        Network object
    """
    net = Network(
        height=height,
        width=width,
        bgcolor="#ffffff",
        font_color="#333333",
        directed=False,
        notebook=True,
        cdn_resources="remote"
    )

    # Bật physics để layout tự động
    net.force_atlas_2based(
        gravity=-80,
        central_gravity=0.01,
        spring_length=200,
        spring_strength=0.05,
        damping=0.4,
        overlap=0.5
    )

    if isolated_nodes is None:
        isolated_nodes = set()

    # ---- Tính thuộc tính trực quan ----
    if edges_df.height > 0:
        scores = edges_df["score"].to_list()
        min_score_val = min(scores)
        max_score_val = max(scores)
        score_range = max_score_val - min_score_val if max_score_val > min_score_val else 1.0
    else:
        min_score_val = 0
        max_score_val = 1
        score_range = 1.0

    # Đếm số cạnh nối vào mỗi node (để tính kích thước node)
    node_edge_count: dict[str, int] = {}
    node_total_score: dict[str, float] = {}
    for row in edges_df.iter_rows(named=True):
        for cat in [row["cat_a"], row["cat_b"]]:
            node_edge_count[cat] = node_edge_count.get(cat, 0) + 1
            node_total_score[cat] = node_total_score.get(cat, 0) + row["score"]

    # ---- Thêm connected nodes ----
    all_connected_nodes = set(edges_df["cat_a"].to_list()) | set(edges_df["cat_b"].to_list())
    max_edge_count = max(node_edge_count.values()) if node_edge_count else 1

    # Bảng màu node — tông xanh dương nhẹ theo reference screenshot
    for node in sorted(all_connected_nodes):
        n_edges = node_edge_count.get(node, 0)
        total_score = node_total_score.get(node, 0)

        # Kích thước node tỉ lệ với số cạnh
        size_ratio = n_edges / max_edge_count
        node_size = 15 + 35 * size_ratio

        # Màu node — tông xanh dương, đậm hơn nếu nhiều cạnh hơn
        blue_intensity = int(180 + 75 * (1 - size_ratio))  # 180–255
        node_color = f"rgba(100, {int(170 + 50*(1-size_ratio))}, {blue_intensity}, 0.85)"

        # Tooltip cho node
        tooltip = (
            f"📦 {node}\n"
            f"Số cạnh: {n_edges}\n"
            f"Tổng score: {total_score:.4f}"
        )

        net.add_node(
            node,
            label=node,
            title=tooltip,
            size=node_size,
            color=node_color,
            font={"size": 14, "face": "Arial", "color": "#333333"},
            borderWidth=2,
            borderWidthSelected=3,
        )

    # ---- Thêm isolated nodes (Sữa, Tã) ----
    for node in sorted(isolated_nodes):
        if node not in all_connected_nodes:  # Tránh thêm trùng
            net.add_node(
                node,
                label=node,
                title=f"📦 {node}\n(Node cô lập — đã loại cạnh)",
                size=20,
                color="rgba(200, 200, 200, 0.6)",  # Màu xám nhạt
                font={"size": 14, "face": "Arial", "color": "#999999"},
                borderWidth=1,
                borderWidthSelected=2,
            )

    # ---- Thêm edges ----
    for row in edges_df.iter_rows(named=True):
        score = row["score"]
        normalized = (score - min_score_val) / score_range  # 0–1

        # Độ dày cạnh: 1–15 pixel
        edge_width = 1 + 14 * normalized

        # Màu cạnh: xanh dương nhạt → xanh dương đậm
        r = int(150 - 120 * normalized)
        g = int(200 - 120 * normalized)
        b = int(230 - 70 * normalized)
        alpha = 0.3 + 0.6 * normalized
        edge_color = f"rgba({r}, {g}, {b}, {alpha})"

        # Tooltip cho cạnh
        tooltip = (
            f"🔗 {row['cat_a']} ↔ {row['cat_b']}\n"
            f"─────────────────────────\n"
            f"count(A,B) = {row['count_ab']:,}\n"
            f"count(A)   = {row['count_a']:,}\n"
            f"count(B)   = {row['count_b']:,}\n"
            f"P(B|A)     = {row['p_b_given_a']:.4f}\n"
            f"P(A|B)     = {row['p_a_given_b']:.4f}\n"
            f"Score      = {row['score']:.4f}"
        )

        net.add_edge(
            row["cat_a"],
            row["cat_b"],
            value=edge_width,
            title=tooltip,
            color=edge_color,
            width=edge_width,
        )

    return net

print("✅ Đã định nghĩa hàm build_pyvis_graph()")

✅ Đã định nghĩa hàm build_pyvis_graph()


In [56]:
# ============================================================
# 6.2 Tạo 2 đồ thị HTML
# ============================================================

# Thư mục lưu file HTML tạm
output_dir = Path(".")  # Lưu cùng thư mục notebook

# --- Đồ thị L1 ---
print("📊 Đang xây dựng đồ thị Category L1...")
net_l1 = build_pyvis_graph(
    edges_l1,
    title="Category L1 Graph",
    isolated_nodes=ISOLATED_CATEGORIES
)
l1_html_path = output_dir / "graph_l1.html"
net_l1.save_graph(str(l1_html_path))
print(f"   Đã lưu: {l1_html_path}")

# --- Đồ thị L1+L2 ---
print("📊 Đang xây dựng đồ thị Category L1+L2...")
net_l1l2 = build_pyvis_graph(
    edges_l1l2,
    title="Category L1+L2 Graph",
    isolated_nodes=ISOLATED_CATEGORIES
)
l1l2_html_path = output_dir / "graph_l1l2.html"
net_l1l2.save_graph(str(l1l2_html_path))
print(f"   Đã lưu: {l1l2_html_path}")

📊 Đang xây dựng đồ thị Category L1...
   Đã lưu: graph_l1.html
📊 Đang xây dựng đồ thị Category L1+L2...
   Đã lưu: graph_l1l2.html


In [57]:
# ============================================================
# 6.3 Hiển thị đồ thị với tab chuyển đổi
# ============================================================

# Đọc nội dung HTML của 2 đồ thị
with open(l1_html_path, "r", encoding="utf-8") as f:
    l1_html_content = f.read()

with open(l1l2_html_path, "r", encoding="utf-8") as f:
    l1l2_html_content = f.read()

# Escape dấu nháy đơn cho srcdoc
l1_escaped = l1_html_content.replace("'", "&#39;").replace("\n", " ")
l1l2_escaped = l1l2_html_content.replace("'", "&#39;").replace("\n", " ")

# Tạo giao diện tab chuyển đổi
tab_html = f"""
<div id="graph-container" style="font-family: 'Segoe UI', Arial, sans-serif;">
    <!-- Tiêu đề -->
    <div style="margin-bottom: 12px; padding: 8px 0;">
        <span style="font-size: 14px; color: #555; font-weight: 500;">Chọn loại đồ thị</span>
    </div>

    <!-- Tab buttons -->
    <div style="display: flex; gap: 0; margin-bottom: 0; border-bottom: 2px solid #e0e0e0;">
        <button
            id="btn-l1"
            onclick="switchTab('l1')"
            style="
                padding: 10px 24px;
                border: none;
                background: #4a90d9;
                color: white;
                font-size: 14px;
                font-weight: 600;
                cursor: pointer;
                border-radius: 8px 8px 0 0;
                transition: all 0.2s;
                outline: none;
            "
        >
            ● Category L1 Graph
        </button>
        <button
            id="btn-l1l2"
            onclick="switchTab('l1l2')"
            style="
                padding: 10px 24px;
                border: none;
                background: #e8e8e8;
                color: #666;
                font-size: 14px;
                font-weight: 600;
                cursor: pointer;
                border-radius: 8px 8px 0 0;
                transition: all 0.2s;
                outline: none;
            "
        >
            ○ Category L1+L2 Graph
        </button>
    </div>

    <!-- Graph iframes -->
    <div style="border: 1px solid #e0e0e0; border-top: none; border-radius: 0 0 8px 8px; overflow: hidden;">
        <iframe
            id="frame-l1"
            srcdoc='{l1_escaped}'
            style="width: 100%; height: 750px; border: none; display: block;"
        ></iframe>
        <iframe
            id="frame-l1l2"
            srcdoc='{l1l2_escaped}'
            style="width: 100%; height: 750px; border: none; display: none;"
        ></iframe>
    </div>
</div>

<script>
function switchTab(tab) {{
    var frameL1 = document.getElementById('frame-l1');
    var frameL1L2 = document.getElementById('frame-l1l2');
    var btnL1 = document.getElementById('btn-l1');
    var btnL1L2 = document.getElementById('btn-l1l2');

    if (tab === 'l1') {{
        frameL1.style.display = 'block';
        frameL1L2.style.display = 'none';
        btnL1.style.background = '#4a90d9';
        btnL1.style.color = 'white';
        btnL1.innerHTML = '● Category L1 Graph';
        btnL1L2.style.background = '#e8e8e8';
        btnL1L2.style.color = '#666';
        btnL1L2.innerHTML = '○ Category L1+L2 Graph';
    }} else {{
        frameL1.style.display = 'none';
        frameL1L2.style.display = 'block';
        btnL1.style.background = '#e8e8e8';
        btnL1.style.color = '#666';
        btnL1.innerHTML = '○ Category L1 Graph';
        btnL1L2.style.background = '#4a90d9';
        btnL1L2.style.color = 'white';
        btnL1L2.innerHTML = '● Category L1+L2 Graph';
    }}
}}
</script>
"""

display(HTML(tab_html))

---
## 7. Tổng kết

Notebook đã hoàn thành trực quan hóa đồ thị cross-sell giữa các nhóm danh mục:

1. **Category L1 Graph** — Mối quan hệ mua chéo giữa các danh mục cấp 1 (trừ Sữa và Tã)
2. **Category L1+L2 Graph** — Mối quan hệ chi tiết hơn ở cấp L1+L2

### Node cô lập: Sữa và Tã
- Sữa và Tã vẫn hiển thị trên đồ thị dưới dạng **node xám cô lập**
- Tất cả cạnh liên quan đã bị loại bỏ để đồ thị tập trung vào các mối quan hệ giữa các danh mục còn lại
- Lý do: Sữa và Tã chiếm tỉ trọng quá lớn, tạo ra quá nhiều cạnh mạnh, khiến đồ thị bị "nuốt" bởi 2 danh mục này

### Cách đọc đồ thị
- **Node lớn hơn** = danh mục có nhiều mối liên hệ cross-sell hơn
- **Cạnh dày và đậm** = mối liên hệ mua chéo mạnh (score cao)
- **Cạnh mỏng và nhạt** = mối liên hệ yếu hơn
- **Node xám** = danh mục cô lập (Sữa, Tã — đã loại cạnh)
- **Hover** vào cạnh để xem chi tiết các chỉ số (count, P(B|A), P(A|B), score)
- **Kéo thả** node để sắp xếp lại layout
- **Zoom** để phóng to/thu nhỏ

### Ý nghĩa kinh doanh
- Các cạnh mạnh gợi ý cơ hội cross-sell hiệu quả giữa các danh mục không phải Sữa/Tã
- Sữa và Tã có chiến lược riêng (Solution 2 — diaper up-sale) nên không cần trong đồ thị cross-sell tổng quan